# Training

for linear regression compare normal GD with GD using mini batch(epoch 10- batch_size 32) with SGD (sophisticated gradient descent batch_size 1)

In [61]:
import numpy as np

rng = np.random.default_rng(0)
n = 400
X = np.column_stack([np.ones(n), rng.normal(size=(n, 2))])
true_w = np.array([4.0, 2.5, -1.5])
y = X@ true_w + rng.normal(0, 0.5, n)

In [62]:
print(X)


[[ 1.          0.12573022 -0.13210486]
 [ 1.          0.64042265  0.10490012]
 [ 1.         -0.53566937  0.36159505]
 ...
 [ 1.          0.78820724 -1.08175195]
 [ 1.         -0.50256659 -1.03806416]
 [ 1.         -1.29102854  0.10186287]]


In [64]:


rng = np.random.default_rng(0)
n = 400
X = np.column_stack([np.ones(n), rng.normal(size=(n, 2))])
true_w = np.array([4.0, 2.5, -1.5])
y = X @ true_w + rng.normal(0, 0.5, n)

def descend(lr, steps=200, record=()):
    w = np.zeros(3)
    hist = []
    for s in range(steps):
        pred = X @ w
        grad = 2 * X.T @ (pred - y) / n     # d/dw of mean squared error
        w = w - lr * grad
        if s in record:
            hist.append((s, ((X @ w - y) ** 2).mean(), w.copy()))
    return w, ((X @ w - y) ** 2).mean(), hist

print("the loop, in full:")
print("    pred = X @ w")
print("    grad = 2 * X.T @ (pred - y) / n")
print("    w    = w - lr * grad")
print()

w, mse, hist = descend(0.1, record=(0, 1, 2, 5, 20, 50, 199))
print("lr = 0.1, watching it arrive:")
print("%8s %12s %s" % ("step", "MSE", "weights"))
for s, m, ww in hist:
    print("%8d %12.6f %s" % (s, m, np.round(ww, 4)))
print("  true weights:", true_w)
print()

the loop, in full:
    pred = X @ w
    grad = 2 * X.T @ (pred - y) / n
    w    = w - lr * grad

lr = 0.1, watching it arrive:
    step          MSE weights
       0    15.054674 [ 0.7543  0.4434 -0.2681]
       1    10.033286 [ 1.3651  0.8083 -0.4863]
       2     6.711738 [ 1.8598  1.1085 -0.664 ]
       5     2.097902 [ 2.8481  1.7246 -1.0223]
      20     0.217009 [ 3.9266  2.4461 -1.4225]
      50     0.213096 [ 3.9758  2.4836 -1.4415]
     199     0.213096 [ 3.9759  2.4837 -1.4416]
  true weights: [ 4.   2.5 -1.5]



In [70]:

print("the closed form agrees, because this problem has one:")
exact = np.linalg.solve(X.T @ X, X.T @ y)
print("  gradient descent:", np.round(w, 6))
print("  normal equation :", np.round(exact, 6))
print("  (a neural network has no closed form. that is the only reason we")
print("   iterate at all.)")
print()

print("now the learning rate, which is the whole game:")
print("%10s %14s %16s %s" % ("lr", "MSE @ 200", "distance to w*", "verdict"))
for lr in (0.001, 0.01, 0.1, 0.5, 0.9, 1.05):
    w2, m2, _ = descend(lr)
    d = np.linalg.norm(w2 - exact)
    if not np.isfinite(m2) or m2 > 1e6:
        verdict = "DIVERGED"
    elif d > 1.0:
        verdict = "too slow, still travelling"
    elif d > 1e-4:
        verdict = "nearly there"
    else:        verdict = "converged"
    print("%10.3f %14s %16s %s"
          % (lr, "%.6f" % m2 if np.isfinite(m2) and m2 < 1e9 else "overflow",
             "%.6f" % d if np.isfinite(d) and d < 1e9 else "-", verdict))
print()
print("too small and 200 steps are not enough to arrive. too large and each")
print("step overshoots further than the last, so the loss grows without")
print("bound. there is a real threshold here, not a matter of taste:")
eig = np.linalg.eigvalsh(2 * X.T @ X / n).max()
print("  divergence starts above 2/L, where L is the largest curvature.")
print("  L = %.4f, so the limit is %.4f." % (eig, 2 / eig))
print()

print("the gradient shrinks as you approach the answer, so the steps do too:")
w3 = np.zeros(3)
print("%8s %14s %14s" % ("step", "|gradient|", "step size"))
for s in range(6):
    g = 2 * X.T @ (X @ w3 - y) / n
    print("%8d %14.6f %14.6f" % (s, np.linalg.norm(g), 0.1 * np.linalg.norm(g)))
    w3 = w3 - 0.1 * g
print("  that is automatic braking. it is also why a fixed learning rate")
print("  works at all, and why schedulers exist for when it does not.")

the closed form agrees, because this problem has one:
  gradient descent: [ 3.975905  2.483688 -1.441556]
  normal equation : [ 3.975905  2.483688 -1.441556]
  (a neural network has no closed form. that is the only reason we
   iterate at all.)

now the learning rate, which is the whole game:
        lr      MSE @ 200   distance to w* verdict
     0.001      10.841048         3.376543 too slow, still travelling
     0.010       0.225246         0.114313 nearly there
     0.100       0.213096         0.000000 converged
     0.500       0.213096         0.000000 converged
     0.900       0.213096         0.000003 converged
     1.050       overflow                - DIVERGED

too small and 200 steps are not enough to arrive. too large and each
step overshoots further than the last, so the loss grows without
bound. there is a real threshold here, not a matter of taste:
  divergence starts above 2/L, where L is the largest curvature.
  L = 2.1550, so the limit is 0.9281.

the gradient shri

In [68]:
np.linalg.eigvalsh(2 * X.T @ X / n)

array([1.85713518, 1.9930292 , 2.15501457])

In [ ]:


rng = np.random.default_rng(0)
n = 400
X = np.column_stack([np.ones(n), rng.normal(size=(n, 2))])
true_w = np.array([4.0, 2.5, -1.5])
y = X @ true_w + rng.normal(0, 0.5, n)

def descend(lr,b1,b2, steps=200, record=()):
    w = np.zeros(3)
    t=1
    v = 0
    m= 0
    hist = []
    vit= []
    for s in range(steps):
        pred = X @ w
        grad = 2 * X.T @ (pred - y) / n     # d/dw of mean squared error
        m = b1*m +  (1-b1)*(grad)
        v=b2*v + (1-b2)*(grad**2)
        m_hat = m / (1 - b1 ** t)
        v_hat = v / (1 - b2 ** t)

        # Adam update
        w = w - lr * m_hat / (np.sqrt(v_hat) + 1e-8)
        if s in record:
            hist.append((s, ((X @ w - y) ** 2).mean(), w.copy()))
            vit.append(v)
    return w, ((X @ w - y) ** 2).mean(), hist, vit

print("the loop, in full:")
print("    pred = X @ w")
print("    grad = 2 * X.T @ (pred - y) / n")
print("    w    = w - lr * grad")
print()

w, mse, hist, vit = descend(0.1,0.9,0.99, record=(0, 1, 2, 5, 20, 50, 199))
print("lr = 0.1, watching it arrive:")
print("%8s %12s %s" % ("step", "MSE", "weights"))
for s, m, ww in hist:
    print("%8d %12.6f %s" % (s, m, np.round(ww, 4)))
print("  true weights:", true_w)
print(vit)

the loop, in full:
    pred = X @ w
    grad = 2 * X.T @ (pred - y) / n
    w    = w - lr * grad

lr = 0.1, watching it arrive:
    step          MSE weights
       0    21.140571 [ 0.1049  0.1049 -0.1049]
       1    19.211399 [ 0.2461  0.246  -0.2458]
       2    17.106287 [ 0.4106  0.4102 -0.4092]
       5    10.979534 [ 0.9841  0.9775 -0.961 ]
      20     0.615064 [ 3.7706  3.0198 -1.6736]
      50     0.238642 [ 4.0982  2.4141 -1.5107]
     199     0.213096 [ 3.976   2.4838 -1.4416]
  true weights: [ 4.   2.5 -1.5]
[array([0.56898163, 0.19661638, 0.07186484]), array([1.10384472, 0.3742925 , 0.13254215]), array([1.59624262, 0.52856943, 0.1798467 ]), array([2.78950935, 0.84347445, 0.24719217]), array([4.07056402, 0.93757091, 0.29886914]), array([3.20258612, 0.77559927, 0.23939398]), array([0.71898182, 0.17401079, 0.05385083])]


b = 0.9
lr = 0.1, watching it arrive:
    step          MSE weights
       0    15.054674 [ 0.7543  0.4434 -0.2681]
       1     5.682226 [ 2.044   1.2074 -0.7276]
       2     0.511887 [ 3.5702  2.1244 -1.274 ]
       5    10.663904 [ 6.7085  4.1465 -2.4249]
      20     2.658796 [ 5.2503  3.3614 -1.9244]
      50     0.274579 [ 4.1258  2.6786 -1.5153]
     199     0.213096 [ 3.9758  2.4836 -1.4415]
b = 0.5
lr = 0.3, watching it arrive:
    step          MSE weights
       0     4.562682 [ 2.2629  1.3302 -0.8042]
       1     0.388397 [ 4.3656  2.619  -1.5622]
       2     2.242338 [ 5.1903  3.1984 -1.8743]
       5     0.269258 [ 3.7581  2.3995 -1.3719]
      20     0.213098 [ 3.9746  2.4831 -1.4411]
      50     0.213096 [ 3.9759  2.4837 -1.4416]
     199     0.213096 [ 3.9759  2.4837 -1.4416]

lr = 0.9, watching it arrive:
    step          MSE weights
       0     4.562682 [ 2.2629  1.3302 -0.8042]
       1     2.385455 [ 5.2708  3.1511 -1.8839]
       2    14.978500 [ 7.2345  4.4423 -2.6093]
       5     1.712174 [ 2.8501  2.0484 -1.0834]
      20     0.823803 [ 4.3767  3.1469 -1.6632]
      50     0.302717 [ 4.1548  2.7177 -1.5377]
     199     0.213096 [ 3.9758  2.4836 -1.4415]
  true weights: [ 4.   2.5 -1.5]

In [101]:
w**2

array([15.80847339,  6.16915914,  2.07817947])

In [102]:
w

array([ 3.9759871 ,  2.48377921, -1.44158922])

In [1]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier

# Features: [age, income]
X = np.array([
    [18, 2000],
    [22, 2500],
    [25, 3000],
    [28, 4000],
    [35, 5000],
    [42, 6000]
])

# 0 = No purchase, 1 = Purchase
y = np.array([0, 0, 0, 1, 1, 1])

model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=2,
    random_state=42
)

model.fit(X, y)

print(model.predict([[30, 4500]]))
print(model.predict_proba([[30, 4500]]))

[1]
[[2.19017313e-05 9.99978098e-01]]


In [2]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

# Features: surface area
X = np.array([[50], [60], [70], [80]])

# Target: house prices (thousands)
y = np.array([100, 120, 140, 160])

model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=1,
    loss="squared_error",
    random_state=42
)

model.fit(X, y)

predictions = model.predict(X)

print(predictions)

[100.03055701 120.0060627  139.99620047 159.96717981]
